# 12 Writing scripts

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part III — From commands to scripts</span>
    <span class="bp-meta">Notebook&nbsp;12</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    Turning a runnable file into a proper, reusable tool: inputs and variables,
    flags, functions, clean formatted output, and failing safely.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. data/ is
# read-only; every script we write and run lives in a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

In Notebook 11 you made a file *runnable* — a shebang, the executable bit, `./`.
But that script did one fixed thing. A real tool is different: you hand it an
input, it names its data in readable variables, organizes its work into functions,
prints clean output, and — when something is wrong — fails early and loudly instead
of limping on with garbage.

This notebook grows exactly that. We start from "a few commands in a file" and, one
idea at a time, build up to `analyze.sh` — a script that takes a trajectory file
and a flag, extracts and averages its energies, prints a formatted result, and
writes a summary file, all under a header that makes it fail safely. Everything you
have learned in Part III converges here.

One boundary, stated up front. This notebook's script is **linear**: one input →
process → output. Making a script *decide* (with `if`) and *repeat* (loop over many
files) is the next notebook. So our capstone processes **one** trajectory; Notebook
13 will make it sweep a whole directory of them. (As ever — the data is a
playground, **no physics required.**)

## A. Anatomy: a script is commands in a file, with a safe header

A script is nothing mysterious: it is the same commands you type at the prompt,
saved in a file so you can re-run them (Notebook 11). From the very first one,
though, we give every script the same two-line header:

```bash
#!/usr/bin/env bash
set -euo pipefail
```

The first line is the shebang from Notebook 11. The second, `set -euo pipefail`, is
the single most valuable habit in script-writing: it makes the script **fail
safely** — stop the moment something goes wrong, rather than barrel on. We unpack
exactly what it does in §H; for now, treat it as boilerplate that every script
wears, and we will model it throughout. Here is a first script under that header:

```bash
#!/usr/bin/env bash
set -euo pipefail
system="lj38"
echo "analysis script for: $system"
```

In [2]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
cat > scratch/first.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
system="lj38"
echo "analysis script for: $system"
EOF
chmod +x scratch/first.sh

In [3]:
(cd scratch && ./first.sh)

analysis script for: lj38


That is the shape of every script below: the header, then the work.

## B. Variables: naming your data

A variable gives a name to a value, so the script reads like a description of what
it does instead of a wall of literals. Assignment has one famous trap — **no spaces
around the `=`**:

In [4]:
system="lj38"; echo "running analysis for $system"

running analysis for lj38


Put a space around the `=` and the shell reads the first word as a *command* name,
not an assignment — a confusing error the first time you meet it:

In [5]:
system = "lj38" 2>/dev/null || echo "with spaces, bash thinks 'system' is a command to run — assignment needs NONE around ="

with spaces, bash thinks 'system' is a command to run — assignment needs NONE around =


You *use* a variable by writing `$name` — and, per Notebook 10, you **quote it on
use**: `"$name"`. The everyday forms:

<div class="bp-card">
  <span class="bp-card-cmd">Variables</span> — <span class="bp-card-job">naming values (shell syntax, not a command). Quote on use; assign with no spaces around <code>=</code>.</span>
  <table>
    <tr><td>name=value</td><td>assign — <b>no spaces</b> around the <code>=</code></td></tr>
    <tr><td>"$name"</td><td>use the value — always quote on use (Notebook 10)</td></tr>
    <tr><td>"${name:-default}"</td><td>use <i>default</i> if name is unset or empty (Notebook 10)</td></tr>
    <tr><td>local name=value</td><td>a variable confined to its function (§E)</td></tr>
    <tr><td>name="$(cmd)"</td><td>capture a command's output into the value (Notebook 10)</td></tr>
  </table>
</div>

(These are *script-local* variables. Making a variable part of the **environment**,
so other programs see it — `export` — belongs to Notebook 14.)

## C. Positional parameters: acting on what you pass

A tool earns its keep by working on *whatever you give it*. The arguments after the
script's name arrive as the **positional parameters** `$1`, `$2`, and so on — with
`$#` counting them and `"$@"` standing for all of them:

<div class="bp-card">
  <span class="bp-card-cmd">Positional parameters</span> — <span class="bp-card-job">the arguments a script (or function) was called with.</span>
  <table>
    <tr><td>$0</td><td>the script's own name</td></tr>
    <tr><td>$1&nbsp;&nbsp;$2&nbsp;…</td><td>the first, second, … argument</td></tr>
    <tr><td>$#</td><td>how many arguments were passed</td></tr>
    <tr><td>"$@"</td><td><b>all</b> arguments, each kept as a separate word — quote it (Notebook 10)</td></tr>
    <tr><td>$*</td><td>all arguments mashed into <b>one</b> word — rarely what you want</td></tr>
    <tr><td>shift</td><td>drop <code>$1</code>; <code>$2</code> becomes <code>$1</code>, and so on</td></tr>
  </table>
</div>

Here is a script that simply reports what it received:

```bash
#!/usr/bin/env bash
set -euo pipefail
echo "script name: $0"
echo "first arg:   $1"
echo "arg count:   $#"
echo "all args:    $@"
```

In [6]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
cat > scratch/args.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
echo "script name: $0"
echo "first arg:   $1"
echo "arg count:   $#"
echo "all args:    $@"
EOF
chmod +x scratch/args.sh

In [7]:
(cd scratch && ./args.sh lj38.xyz pt-slab.xyz)

script name: ./args.sh


first arg:   lj38.xyz


arg count:   2


all args:    lj38.xyz pt-slab.xyz


The script now does different work depending on its input — the difference between a
one-off and a tool.

## D. Flags via `getopts`: options, the standard way

Arguments are positional; **flags** (`-v`, `-n 5`) let a caller switch behaviour on
and off by name. Bash has a dedicated helper, **`getopts`**, and it is almost always
written as the same copy-me skeleton:

```bash
verbose=0
n=3
while getopts "vn:" opt; do
  case "$opt" in
    v) verbose=1 ;;          # a plain flag
    n) n="$OPTARG" ;;        # a flag that takes a value
    *) echo "usage: show.sh [-v] [-n N] FILE" >&2; exit 1 ;;
  esac
done
shift $(( OPTIND - 1 ))       # drop the parsed flags; "$1" is now the first real argument
```

The option string `"vn:"` says: accept `-v` (no value) and `-n` (the trailing
**`:`** means it takes a value). For each flag, `getopts` puts the letter in `opt`,
and any value in **`OPTARG`**; **`OPTIND`** tracks how far it got, so the final
`shift` discards the flags and leaves your real arguments as `$1`, `$2`, …

:::{admonition} On the while/case machinery
:class: note
The `while … do … done` loop and the `case … esac` choice are **control flow** —
the subject of Notebook 13. Here, treat the skeleton above as boilerplate to copy
and fill in; you will understand its mechanics fully next notebook. Two curation
notes: `getopts` handles **short** options only — long ones like `--verbose` need
more than it offers, and we leave them out of scope.
:::

Here that skeleton is, wired into a small script that shows the first *N* lines of a
file, with `-v` relabelling the output:

In [8]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'line one\nline two\nline three\nline four\n' > scratch/notes.txt
cat > scratch/show.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
label="quiet"
n=3
while getopts "vn:" opt; do
  case "$opt" in
    v) label="verbose" ;;
    n) n="$OPTARG" ;;
    *) echo "usage: show.sh [-v] [-n N] FILE" >&2; exit 1 ;;
  esac
done
shift $(( OPTIND - 1 ))
file="$1"
echo "[$label] first $n lines of $file:"
head -n "$n" "$file"
EOF
chmod +x scratch/show.sh

In [9]:
(cd scratch && ./show.sh -v -n 2 notes.txt)

[verbose] first 2 lines of notes.txt:


line one


line two


Both flags took effect: `-v` switched the label to `verbose`, and `-n 2` showed two
lines instead of the default three.

## E. Functions: naming a piece of work

When a script repeats a step — or just grows long enough to need sections — you give
that piece of work a name. A **function** is a mini-script inside your script, with
its **own** `$1`, `$2`, and its own `local` variables:

<div class="bp-card">
  <span class="bp-card-cmd">Functions</span> — <span class="bp-card-job">name a reusable piece of work; it gets its own arguments.</span>
  <table>
    <tr><td>name() { …; }</td><td>define a function</td></tr>
    <tr><td>name arg1 arg2</td><td>call it — inside, <code>$1</code> <code>$2</code> are <b>its</b> arguments</td></tr>
    <tr><td>local v=…</td><td>a variable local to the function, not leaking out</td></tr>
    <tr><td>return N</td><td>set the function's <b>exit code</b> (0–255) — not a value</td></tr>
    <tr><td>echo … → "$(name)"</td><td>to hand back <b>data</b>, echo it and capture with <code>$( )</code></td></tr>
  </table>
</div>

The one thing that trips newcomers: a function's `return` is an **exit code**, not a
return value. `return 0` means "success," not "the answer is zero":

In [10]:
is_even() { local n="$1"; return $(( n % 2 )); }
is_even 4; echo "exit code from is_even 4: $?  (0 = success = yes, even)"

exit code from is_even 4: 0  (0 = success = yes, even)


So to hand **data** back from a function, you `echo` it and **capture** it with
`$(…)` at the call site — exactly the command-substitution idea from Notebook 10.
Here is a function that averages a log's energies (the `grep`/`awk` from Notebooks 6
and 8) and returns the number:

In [11]:
energy_mean() {
  local file="$1"
  grep 'Total FORCE_EVAL' "$file" | grep -oE '\-[0-9]+\.[0-9]+' | awk '{ s += $1; n++ } END { printf "%.4f\n", s/n }'
}
m="$(energy_mean data/logs/gr2hno3-nvt.log)"
echo "captured mean = $m eV"

captured mean = -143.4444 eV


The `local file` stays inside the function; the result comes back through `echo` +
`$(…)`. That is the pattern for every "compute something and give it back."

## F. `printf`: clean, formatted output

Scripts report results, and for that `printf` beats `echo`. It takes a **format
string** with placeholders, then the values to drop in — so columns line up and
numbers carry the precision you choose.

```{command-card} printf
```

In [12]:
printf '%-8s %8.3f eV\n' "lj38" 3.14159

lj38        3.142 eV


`%-8s` is a string left-justified in an 8-wide field; `%8.3f` is a float right-
justified in 8 columns, three decimals; `\n` is the newline you add yourself. The
one behaviour that surprises everyone: the format string is **reused** for any
extra arguments, which makes printing a list a one-liner:

In [13]:
printf '%s\n' alpha beta gamma

alpha


beta


gamma


Three arguments, one `%s\n` template, three lines. (This is *why* `printf` is the
scripting choice over `echo`: no `-e`/`-n` portability guesswork — the format string
says exactly what you get.)

## G. Heredocs: generating multi-line text and files

Sometimes a script needs to emit a whole block — a config file, a report, a
submission script. A **here-document** feeds an inline block to a command, and is
the cleanest way to write a multi-line file:

<div class="bp-card">
  <span class="bp-card-cmd">Heredocs &amp; here-strings</span> — <span class="bp-card-job">feed an inline block (or one line) to a command's input.</span>
  <table>
    <tr><td>cmd &lt;&lt;EOF … EOF</td><td>feed the block to <i>cmd</i>; <code>$vars</code> and <code>$(…)</code> <b>are</b> expanded</td></tr>
    <tr><td>cmd &lt;&lt;'EOF' … EOF</td><td>quoted delimiter: the block is <b>literal</b> — no expansion</td></tr>
    <tr><td>cmd &lt;&lt;-EOF … EOF</td><td>the <code>-</code> strips leading <b>tabs</b>, so you can indent the block</td></tr>
    <tr><td>cmd &lt;&lt;&lt; "text"</td><td>here-<i>string</i>: feed a single line as input</td></tr>
  </table>
</div>

With a plain `EOF`, the block behaves like a double-quoted string — variables and
command substitutions expand:

In [14]:
proj="lj38"; cat <<EOF
project: $proj
built:   $(date +%F)
EOF

project: lj38


built:   2026-06-10


**Quote** the delimiter — `<<'EOF'` — and the block is taken literally, expanding
nothing (just like single quotes, Notebook 10) — what you want when the text itself
contains `$`:

In [15]:
cat <<'EOF'
literal: $proj and $(date +%F) are NOT expanded here
EOF

literal: $proj and $(date +%F) are NOT expanded here


The "killer app" is writing a file: redirect the heredoc and you have generated a
config in place. This is exactly how you will produce input decks and (Notebook 16)
cluster submission scripts:

In [16]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [17]:
proj="lj38"; cat > scratch/run.inp <<EOF
&GLOBAL
  PROJECT $proj
  RUN_TYPE ENERGY
&END
EOF
cat scratch/run.inp

&GLOBAL


  PROJECT lj38


  RUN_TYPE ENERGY


&END


And the one-line cousin, the **here-string** `<<<`, feeds a single string to a
command's input — handy for piping a variable in without an `echo`:

In [18]:
wc -w <<< "one two three four"

4


## H. Robustness: failing early and loudly

Now we can open up the header we have used all along. `set -euo pipefail` is three
switches that turn silent, limping failure into an immediate, visible stop:

<div class="bp-card">
  <span class="bp-card-cmd">set — the safety switches</span> — <span class="bp-card-job">put <code>set -euo pipefail</code> at the top of every script.</span>
  <table>
    <tr><td>set -e</td><td>exit immediately if any command fails</td></tr>
    <tr><td>set -u</td><td>treat use of an <b>unset</b> variable as an error (pair with <code>"${v:-default}"</code>)</td></tr>
    <tr><td>set -o pipefail</td><td>a pipeline fails if <b>any</b> stage fails, not only the last</td></tr>
    <tr><td>set -euo pipefail</td><td>the standard one-line header — all three at once</td></tr>
    <tr><td>bash -x script&nbsp;&nbsp;/&nbsp;&nbsp;set -x</td><td>trace each command as it runs (debugging)</td></tr>
  </table>
</div>

`set -u` is the one with a direct line back to Notebook 10: a typo'd or never-set
variable is no longer silently empty — it stops the script. Watch a script die the
instant it reaches an unset variable (the `|| echo` is only so this page keeps
running):

In [19]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
cat > scratch/strict.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
echo "starting"
echo "label is: $label"
echo "finishing"
EOF
chmod +x scratch/strict.sh

In [20]:
(cd scratch && ./strict.sh) 2>&1 || echo "↳ set -u stopped the script: 'label' was never set (note 'finishing' never printed)"

starting


./strict.sh: line 4: label: unbound variable


↳ set -u stopped the script: 'label' was never set (note 'finishing' never printed)


It printed `starting`, hit the unset `label`, and **stopped** — it never reached
`finishing`. That is the whole point: a broken run ends where it broke, loudly,
instead of producing a wrong answer. The fix is the Notebook-10 default value:

In [21]:
cat > scratch/strict-ok.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
echo "starting"
echo "label is: ${label:-default}"
echo "finishing"
EOF
chmod +x scratch/strict-ok.sh

In [22]:
(cd scratch && ./strict-ok.sh)

starting


label is: default


finishing


And when a script misbehaves and you want to *see* what it is doing, **`bash -x`**
traces every line (each `+` is a command as the shell runs it):

In [23]:
bash -x scratch/strict-ok.sh 2>&1 | head -n 7

+ set -euo pipefail


+ echo starting


starting


+ echo 'label is: default'


label is: default


+ echo finishing


finishing


The principle behind the whole header: **fail early and loudly.** A script that
stops at the first sign of trouble is far kinder than one that quietly writes a
corrupt file you discover three weeks later.

## Exercises

One script grows across this set, from a fixed greeting to a real analysis tool.
Everything is created and run in a fresh `scratch/`; `data/` stays read-only. (On
this page the scripts are written with here-documents so they are reproducible; in
your own terminal you would type them into Vim from Notebook 9 — the result is the
same file.)

### Warm-up 1 (worked) — A first script with variables

Under the safe header, assign a couple of variables and use them, then run with
`./`.

In [24]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [25]:
# (solution hidden on the public site)


system: lj38 (binding-energy run)


In [26]:
out="$(cd "$ROOT/scratch" && ./greet.sh)"
check '[ -x "$ROOT/scratch/greet.sh" ] && printf "%s" "$out" | grep -q "system: lj38 (binding-energy run)"' \
      "the script runs under ./ and prints the variables it was given"

✓ the script runs under ./ and prints the variables it was given


### Warm-up 2 (your turn) — Act on an argument

Write a script that takes a filename as `$1`, and reports both how many arguments it
got (`$#`) and the name it was handed (`"$1"`).

In [27]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [28]:
# (solution hidden on the public site)


you passed 1 argument(s)


the file is: sample.xyz


In [29]:
out="$(cd "$ROOT/scratch" && ./describe.sh sample.xyz)"
check 'printf "%s" "$out" | grep -q "1 argument" && printf "%s" "$out" | grep -q "the file is: sample.xyz"' \
      "the script acted on its argument: one passed, and its name reported"

✓ the script acted on its argument: one passed, and its name reported


### Applied 1 (your turn) — Add a flag with `getopts`

Extend a script with the `getopts` skeleton: a `-v` flag that changes a label, and a
`-n N` flag that sets how many lines of the file to show. Run it on a scratch file
with `-v -n 2`.

In [30]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'line one\nline two\nline three\nline four\n' > scratch/notes.txt

In [31]:
# (solution hidden on the public site)


[verbose] first 2 lines of notes.txt:


line one


line two


In [32]:
out="$(cd "$ROOT/scratch" && ./peek.sh -v -n 2 notes.txt)"
body="$(printf "%s\n" "$out" | grep -c "^line")"
check 'printf "%s" "$out" | grep -q "\[verbose\]" && [ "$body" -eq 2 ]' \
      "both flags took effect: -v relabelled the output and -n 2 showed two lines"

✓ both flags took effect: -v relabelled the output and -n 2 showed two lines


### Applied 2 (your turn) — Factor a function

Move the energy-averaging step into a **function** with a `local` variable, and have
it return the number via `echo` + `$(…)`. The script takes the log as `$1`.

In [33]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
cp data/logs/gr2hno3-nvt.log scratch/run.log

In [34]:
# (solution hidden on the public site)


mean energy: -143.4444


In [35]:
out="$(cd "$ROOT/scratch" && ./mean.sh run.log)"
check '[ "$out" = "mean energy: -143.4444" ]' \
      "the function computed the mean and handed it back through echo + \$( )"

✓ the function computed the mean and handed it back through echo + $( )


### Applied 3 (worked) — `printf` and a heredoc

Format a result line with `printf`, then generate a small summary file with a
heredoc (note the inner delimiter is unquoted, so the variables expand into the
file).

In [36]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [37]:
# (solution hidden on the public site)


lj38     mean = -143.4444 eV


In [38]:
cat scratch/summary.txt

system: lj38


mean energy: -143.4444 eV


In [39]:
out="$(cd "$ROOT/scratch" && ./report.sh)"
check 'printf "%s" "$out" | grep -q "lj38     mean = -143.4444 eV" && [ -f "$ROOT/scratch/summary.txt" ] && grep -q "mean energy: -143.4444 eV" "$ROOT/scratch/summary.txt"' \
      "printf formatted the line and the heredoc wrote the summary file with the values expanded"

✓ printf formatted the line and the heredoc wrote the summary file with the values expanded


### Composite — putting it together (a complete analysis tool)

The capstone, and the payoff of all of Part III. Write `analyze.sh` that takes a
trajectory log as `$1` and an optional `-v` flag; uses a **function** to extract and
average its energies (`grep`/`awk`); prints a `printf`-formatted result; writes a
heredoc **summary file**; and runs under the `set -euo pipefail` header — invoked
with `./`. It composes Notebooks 9–11 (write, quote, run) with 6 and 8 (extract) and
everything in this notebook. It still processes **one** file — looping over many is
the next notebook.

In [40]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
cp data/logs/gr2hno3-nvt.log scratch/run.log

In [41]:
# (solution hidden on the public site)


verbose result mean energy = -143.4444 eV


In [42]:
cat scratch/summary.txt

file: run.log


mean energy (eV): -143.4444


In [43]:
out="$(cd "$ROOT/scratch" && ./analyze.sh -v run.log)"
check 'printf "%s" "$out" | grep -q "mean energy = -143.4444 eV" && [ -f "$ROOT/scratch/summary.txt" ] && grep -q "mean energy (eV): -143.4444" "$ROOT/scratch/summary.txt"' \
      "analyze.sh ran end to end: flag, function, printf result, and a written summary file"

✓ analyze.sh ran end to end: flag, function, printf result, and a written summary file


### Optional stretch (your turn) — Make it fail loudly

Feel the safety net. Write a script that references an **unset** variable through a
`"${var:-default}"`, so it survives `set -u`; then trace a run with `bash -x` to
watch each step.

In [44]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [45]:
# (solution hidden on the public site)


+ set -euo pipefail


+ echo 'temperature: 300 K'


temperature: 300 K


In [46]:
out="$(cd "$ROOT/scratch" && ./fragile.sh)"
check 'printf "%s" "$out" | grep -q "temperature: 300 K"' \
      "the default value filled in for the unset variable, so the script survived set -u"

✓ the default value filled in for the unset variable, so the script survived set -u


## Outlook

Your script now takes inputs, names its data, factors work into functions, prints
clean output, generates files, and fails safely — a real, reusable tool. But it
still does one thing, once. Next (Notebook 13): **control flow** — `if`/`case`, exit
codes and `&&`/`||`, and loops — so a script can **decide** and **repeat**. That is
what turns `analyze.sh` from a one-file tool into one that sweeps an entire directory
of trajectories in a single run.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to write and run these scripts yourself — nothing to install. The
    published notebooks ship <b>without worked solutions</b>; if you would like
    the reference solutions — to teach from or to check your own work — get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>